# LoRA Fine-Tuning

## Objective

Implemented SFT + LoRA  to fine-tune **TinyLlama-1.1B** on the **databricks-dolly-15k** dataset using LoRA.



#Intsallations

In [ ]:
pip install -U --ignore-installed transformers trl peft datasets accelerate torch -q

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ipython 7.34.0 requires jedi>=0.16, which is not installed.
google-colab 1.0.0 requires pandas==2.2.3, but you have pandas 3.0.6 which is incompatible.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.
libcuml-cu12 26.2.0 requires cuda-toolkit[cublas,cufft,curand,cusolver,cusparse]==12.*, but you have cuda-toolkit 13.0.3.0 which is incompatible.
bigframes 2.48.0 requires rich<14,>=12.4.4, but you have rich 15.0.0 which is incompatible.
cudf-cu12 26.2.1 requires cuda-toolkit[nvcc,nvrtc]==12.*, but you have cuda-toolkit 13.0.3.0 which is incompatible.
cudf-cu12 26.2.1 requires pandas<2.4.0,>=2.0, but you have pandas 3.0.6 which is incompatible.
libraft-cu12 26.2.0 requires cuda-toolkit[cublas,curand,cusolver,cusparse]==12.*, but you have cuda-toolkit 13.0.3.0 which is 

In [ ]:
!pip install -U --force-reinstall \
    torch==2.11.0 \
    torchvision==0.26.0 \
    --index-url https://download.pytorch.org/whl/cu128

Looking in indexes: https://download.pytorch.org/whl/cu128
  Using cached https://download-r2.pytorch.org/whl/cu128/torch-2.11.0%2Bcu128-cp313-cp313-manylinux_2_28_x86_64.whl.metadata (29 kB)
  Using cached https://download-r2.pytorch.org/whl/cu128/torchvision-0.26.0%2Bcu128-cp313-cp313-manylinux_2_28_x86_64.whl.metadata (5.5 kB)
  Using cached filelock-3.32.3-py3-none-any.whl.metadata (2.0 kB)
  Using cached typing_extensions-4.16.0-py3-none-any.whl.metadata (3.3 kB)
  Using cached https://download.pytorch.org/whl/setuptools-78.1.0-py3-none-any.whl.metadata (6.6 kB)
  Using cached sympy-1.14.0-py3-none-any.whl.metadata (12 kB)
  Using cached networkx-3.6.1-py3-none-any.whl.metadata (6.8 kB)
  Using cached https://download.pytorch.org/whl/jinja2-3.1.6-py3-none-any.whl.metadata (2.9 kB)
  Using cached fsspec-2026.7.0-py3-none-any.whl.metadata (10 kB)
  Using cached cuda_bindings-12.9.7-cp313-cp313-manylinux_2_24_x86_64.manylinux_2_28_x86_64.whl.metadata (2.6 kB)
     ━━━━━━━━━━━━━━━━━━━

In [ ]:
import torch
import torchvision

print("PyTorch:", torch.__version__)
print("Torchvision:", torchvision.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

PyTorch: 2.11.0+cu128
Torchvision: 0.26.0+cu128
CUDA available: True
GPU: Tesla T4


In [ ]:
import torch
import gc
import os
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import LoraConfig, PeftModel, get_peft_model
from trl import SFTConfig, SFTTrainer

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

PyTorch version: 2.11.0+cu128
CUDA available: True
GPU: Tesla T4


In [ ]:
!pip show torch torchvision transformers peft trl

Name: torch
Version: 2.11.0+cu128
Summary: Tensors and Dynamic neural networks in Python with strong GPU acceleration
Home-page: https://pytorch.org
Author: 
Author-email: PyTorch Team <packages@pytorch.org>
License: BSD-3-Clause
Location: /usr/local/lib/python3.13/dist-packages
Requires: cuda-bindings, cuda-toolkit, filelock, fsspec, jinja2, networkx, nvidia-cudnn-cu12, nvidia-cusparselt-cu12, nvidia-nccl-cu12, nvidia-nvshmem-cu12, setuptools, sympy, triton, typing-extensions
Required-by: accelerate, bitsandbytes, fastai, peft, sentence-transformers, timm, torchdata, torchvision
---
Name: torchvision
Version: 0.26.0+cu128
Summary: image and video datasets and models for torch deep learning
Home-page: https://github.com/pytorch/vision
Author: PyTorch Core Team
Author-email: soumith@pytorch.org
License: BSD
Location: /usr/local/lib/python3.13/dist-packages
Requires: numpy, pillow, torch
Required-by: fastai, timm
---
Name: transformers
Version: 5.17.0
Summary: Transformers: the model-def

### Utility functions (provided)

In [ ]:
def clear_memory():
    """Free GPU memory between experiments."""
    gc.collect()
    torch.cuda.empty_cache()
    if torch.cuda.is_available():
        print(f"GPU allocated: {torch.cuda.memory_allocated() / 1024**3:.2f} GB")
        print(f"GPU reserved:  {torch.cuda.memory_reserved() / 1024**3:.2f} GB")


def print_trainable_params(model):
    """Display trainable vs total parameter counts."""
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total = sum(p.numel() for p in model.parameters())
    print(f"Trainable parameters: {trainable:>12,}  ({100 * trainable / total:.4f}%)")
    print(f"Total parameters:     {total:>12,}")


def generate_response(model, tokenizer, prompt, max_new_tokens=256):
    """Generate a response from a model given a prompt string."""
    messages = [{"role": "user", "content": prompt}]
    input_text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(input_text, return_tensors="pt").to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=0.7,
            top_p=0.9,
            do_sample=True,
            pad_token_id=tokenizer.pad_token_id,
        )
    response = tokenizer.decode(outputs[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    return response

---
## Part 1 — Configuration

### Task 1: Defineed all hyperparameters

Set the following configuration variables:

| Variable | Value |
|---|---|
| `MODEL_NAME` | `"TinyLlama/TinyLlama-1.1B-Chat-v1.0"` |
| `DATASET_NAME` | `"databricks/databricks-dolly-15k"` |
| `NUM_TRAIN_SAMPLES` | `1000` |
| `MAX_SEQ_LENGTH` | `512` |
| `LORA_R` | `8` |
| `LORA_ALPHA` | `16` |
| `LORA_DROPOUT` | `0.05` |
| `LORA_TARGET_MODULES` | `["q_proj", "v_proj"]` (attention only) |
| `LEARNING_RATE` | `2e-4` |
| `NUM_EPOCHS` | `1` |
| `BATCH_SIZE` | `4` |
| `GRAD_ACCUM_STEPS` | `4` |
| `OUTPUT_DIR` | `"./sft-lora-tinyllama"` |

In [ ]:
# YOUR CODE HERE — define all configuration variables listed above
# ============================================================
# Configuration — change these to experiment
# ============================================================
MODEL_NAME = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"      # Base model (no -Instruct suffix!)
DATASET_NAME = "databricks/databricks-dolly-15k"   # Classic instruction-tuning dataset
NUM_TRAIN_SAMPLES = 1000               # Subset for fast training on Colab

#Maximum number of tokens a Model can process in a single input sequence
MAX_SEQ_LENGTH = 512                   # Maximum sequence length

# LoRA hyperparameters
LORA_R = 8               # Rank
LORA_ALPHA = 16            # Scaling factor (2x rank)
LORA_DROPOUT = 0.05        # Regularization
LORA_TARGET_MODULES = [    # Apply to all linear layers
    "q_proj", "v_proj"
]

# Training hyperparameters
LEARNING_RATE = 2e-4       # 10x higher than full fine-tuning
NUM_EPOCHS = 1
BATCH_SIZE = 4
GRAD_ACCUM_STEPS = 4       # Effective batch size = 4 * 4 = 16
WARMUP_RATIO = 0.03
OUTPUT_DIR = "./sft-lora-qwen"
LOG_DIR = f"{OUTPUT_DIR}/runs"  # TensorBoard log directory

---
## Part 2 — Loaded Model and Tokenizer

### Task 2: Loaded the tokenizer and base model

1. Loaded the tokenizer from `MODEL_NAME`.
2. If the tokenizer has no pad token, set it to the eos token.
3. Load the model in `float16` with `device_map="auto"`.
4. Printed the total parameter count.

In [ ]:
# YOUR CODE HERE — load tokenizer and model
# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# Ensure pad token is set
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.pad_token_id = tokenizer.eos_token_id

# Load the base model
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map="auto",
)

# Count total parameters
total_params = sum(p.numel() for p in model.parameters())

print(f"Vocabulary size: {len(tokenizer):,}")
print(f"Model max length: {tokenizer.model_max_length:,}")
print(f"Pad token: '{tokenizer.pad_token}' (id={tokenizer.pad_token_id})")
print(f"Total parameters: {total_params:,}")

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Vocabulary size: 32,000
Model max length: 2,048
Pad token: '</s>' (id=2)
Total parameters: 1,100,048,384


---
## Part 3 — Test Base Model

### Generated responses from the base model

Used the provided `generate_response` function with these test prompts:
1. `"Explain what a neural network is in 2-3 sentences."`
2. `"What are three tips for better sleep?"`

Stored the responses in a list called `base_responses` for later comparison.

In [ ]:
# YOUR CODE HERE — define test_prompts, generate base model responses, store in base_responses
# Test prompts to evaluate instruction-following ability
test_prompts = [
    "Explain what a neural network is in 2-3 sentences.",
    "What are three tips for better sleep?",

]

print("=" * 70)
print("BASE MODEL RESPONSES (before SFT)")
print("=" * 70)

for i, prompt in enumerate(test_prompts):
    print(f"\n{'─' * 60}")
    print(f"Prompt {i+1}: {prompt}")
    print(f"{'─' * 60}")
    response = generate_response(model, tokenizer, prompt, max_new_tokens=200)
    print(f"Response: {response[:500]}")

# Store base responses for later comparison
base_responses = []
for prompt in test_prompts:
    base_responses.append(generate_response(model, tokenizer, prompt, max_new_tokens=200))

BASE MODEL RESPONSES (before SFT)

────────────────────────────────────────────────────────────
Prompt 1: Explain what a neural network is in 2-3 sentences.
────────────────────────────────────────────────────────────


[transformers] Both `max_new_tokens` (=200) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=200) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Response: A neural network is a type of artificial neural system that is composed of interconnected layers of neurons. These layers process information and make decisions based on input from other layers. The layers are connected by weights, which determine how much information a neuron receives and how much it needs to learn to make a decision. A neural network can be used to solve problems that involve complex patterns or relationships, such as predicting stock prices, identifying diseases, or classifyi

────────────────────────────────────────────────────────────
Prompt 2: What are three tips for better sleep?
────────────────────────────────────────────────────────────


[transformers] Both `max_new_tokens` (=200) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Response: 1. Create a comfortable sleep environment: Ensure that your bedroom is cool, dark, and quiet. This will help you fall asleep faster and stay asleep longer.

2. Establish a bedtime routine: Establish a consistent bedtime routine, such as a bath, reading, or meditation, to help you relax and fall asleep faster.

3. Limit screen time: Avoid using electronic devices, including TVs, computers, and phones, for at least an hour before bedtime. This can help you relax and wind down.

4. Exercise regular


[transformers] Both `max_new_tokens` (=200) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


---
## Part 4 — Prepared the Dataset

### Task 4: Loadd and formated the Dolly dataset

The Dolly dataset has columns: `instruction`, `context`, `response`, `category`.

1. Loaded the dataset (`split="train"`).
2. Writtne a function `format_dolly_to_chat(example)` that converts each example to:
   ```python
   {"messages": [
       {"role": "user", "content": <instruction + context if present>},
       {"role": "assistant", "content": <response>}
   ]}
   ```
   If `context` is non-empty, concatenate it to the instruction with a newline separator.
3. Shuffle with `seed=42`, select `NUM_TRAIN_SAMPLES` examples, and apply the formatting function.
4. Print the size of the formatted dataset and one example.



```
Example formatted entry:
{'messages': [{'content': 'Who were the children of the legendary Garth Greenhand, the High King of the First Men in the series A Song of Ice and Fire?', 'role': 'user'}, {'content': 'Garth the Gardener, John the Oak, Gilbert of the Vines, Brandon of the Bloody Blade, Foss the Archer, Owen Oakenshield, Harlon the Hunter, Herndon of the Horn, Bors the Breaker, Florys the Fox, Maris the Maid, Rose of the Red Lake, Ellyn Ever Sweet, Rowan Gold-Tree', 'role': 'assistant'}]}
```



In [ ]:
# YOUR CODE HERE — load dolly dataset, write format_dolly_to_chat, apply it
# Load Dolly dataset
raw_dataset = load_dataset(DATASET_NAME, split="train")

# Format Dolly examples into chat messages
def format_dolly_to_chat(example):
    if example["context"] and example["context"].strip():
        user_content = f"{example['instruction']}\n{example['context']}"
    else:
        user_content = example["instruction"]

    return {
        "messages": [
            {"role": "user", "content": user_content},
            {"role": "assistant", "content": example["response"]},
        ]
    }

# Shuffle and select the required number of samples
dataset = raw_dataset.shuffle(seed=42).select(range(NUM_TRAIN_SAMPLES))

# Apply formatting
dataset = dataset.map(
    format_dolly_to_chat,
    remove_columns=raw_dataset.column_names,
)

# Print dataset size and example
print(f"Formatted dataset size: {len(dataset):,}")
print("\nExample formatted entry:")
print(dataset[0])

Formatted dataset size: 1,000

Example formatted entry:
{'messages': [{'role': 'user', 'content': 'Who were the children of the legendary Garth Greenhand, the High King of the First Men in the series A Song of Ice and Fire?'}, {'role': 'assistant', 'content': 'Garth the Gardener, John the Oak, Gilbert of the Vines, Brandon of the Bloody Blade, Foss the Archer, Owen Oakenshield, Harlon the Hunter, Herndon of the Horn, Bors the Breaker, Florys the Fox, Maris the Maid, Rose of the Red Lake, Ellyn Ever Sweet, Rowan Gold-Tree'}]}


In [ ]:
# Verify the chat template renders correctly
sample = dataset[100]
rendered = tokenizer.apply_chat_template(sample["messages"], tokenize=False)
print("Rendered chat template (first example):\n")
print(rendered[:501])

Rendered chat template (first example):

<|user|>
Which TV Show is about a zip code in Beverly Hills?</s>
<|assistant|>
90210</s>



---
## Part 5 — Configured LoRA

### Task 5: d the LoRA config and inspect trainable parameters

1. Creatde a `LoraConfig` using the hyperparameters from Task 1. Set `bias="none"` and `task_type="CAUSAL_LM"`.
2. Applied it to the model with `get_peft_model`.
3. Used `print_trainable_params` to display the trainable vs total parameters.
4. Printed the LoRA-wrapped structure of one attention layer (e.g., `q_proj` of layer 0).

In [ ]:
# YOUR CODE HERE — create LoraConfig, apply to model, print trainable params

# Define LoRA configuration
peft_config = LoraConfig(
    r=LORA_R,                           # Rank of decomposition
    lora_alpha=LORA_ALPHA,              # Scaling factor (alpha/r applied)
    lora_dropout=LORA_DROPOUT,          # Dropout for regularization
    bias="none",                        # Don't train bias terms
    task_type="CAUSAL_LM",             # Decoder-only language model
    target_modules=LORA_TARGET_MODULES, # All attention + MLP layers
)

print("LoRA Configuration:")
print(f"  Rank (r):          {peft_config.r}")
print(f"  Alpha (α):         {peft_config.lora_alpha}")
print(f"  Effective scaling: {peft_config.lora_alpha / peft_config.r}")
print(f"  Dropout:           {peft_config.lora_dropout}")
print(f"  Target modules:    {peft_config.target_modules}")
print(f"  Task type:         {peft_config.task_type}")

LoRA Configuration:
  Rank (r):          8
  Alpha (α):         16
  Effective scaling: 2.0
  Dropout:           0.05
  Target modules:    {'v_proj', 'q_proj'}
  Task type:         CAUSAL_LM


In [ ]:
!pip install -U "torchao>=0.16.0"

In [ ]:
import torchao

print("torchao version:", torchao.__version__)

torchao version: 0.18.0


In [ ]:
import torch
!pip install torchao>=0.16.0 # Install compatible torchao version
# Apply LoRA to the model and inspect
lora_model = get_peft_model(model, peft_config)

print("\n" + "=" * 50)
print("PARAMETER COMPARISON")
print("=" * 50)
print_trainable_params(lora_model)

# Show the LoRA module structure for one layer
print("\n\nLoRA module example (layer 0, q_proj):")
print(lora_model.model.model.layers[0].self_attn.q_proj)


PARAMETER COMPARISON
Trainable parameters:    1,126,400  (0.1023%)
Total parameters:     1,101,174,784


LoRA module example (layer 0, q_proj):
lora.Linear(
  (base_layer): Linear(in_features=2048, out_features=2048, bias=False)
  (lora_dropout): ModuleDict(
    (default): Dropout(p=0.05, inplace=False)
  )
  (lora_A): ModuleDict(
    (default): Linear(in_features=2048, out_features=8, bias=False)
  )
  (lora_B): ModuleDict(
    (default): Linear(in_features=8, out_features=2048, bias=False)
  )
  (lora_embedding_A): ParameterDict()
  (lora_embedding_B): ParameterDict()
  (lora_magnitude_vector): ModuleDict()
)


In [ ]:
!pip install -U bitsandbytes

---
## Part 6 — Train

### Task 6: Set up SFTTrainer and ran training

1. Deleted the PEFT-wrapped model and call `clear_memory()`. Reload the base model fresh (SFTTrainer applies LoRA itself via `peft_config`).
2. Created an `SFTConfig` with the hyperparameters from Task 1. Include:
   - Cosine LR scheduler, warmup ratio 0.03, weight decay 0.01
   - `gradient_checkpointing=True`
   - bf16 if supported, else fp16
   - `max_length=MAX_SEQ_LENGTH`
   - Logging every 10 steps
3. Created an `SFTTrainer` with the model, training args, dataset, tokenizer, and `peft_config`.
4. Called `trainer.train()` and print the final loss.

In [ ]:
# Free GPU memory
clear_memory()

# Reload a fresh base model
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map="auto",
)
# Check whether BF16 is supported
bf16_supported = torch.cuda.is_available() and torch.cuda.is_bf16_supported()

# Create SFT training configuration
training_args = SFTConfig(
    output_dir=OUTPUT_DIR,
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM_STEPS,

    gradient_checkpointing=True,

    learning_rate=LEARNING_RATE,
    weight_decay=0.01,

    logging_steps=10,
    save_steps=0.25,

    optim="paged_adamw_8bit",
    lr_scheduler_type="cosine",
    warmup_steps=2,

    report_to="tensorboard",
    seed=42,

    bf16=False,
    fp16=True,

    torch_compile=False,
    dataset_num_proc=os.cpu_count(),

    max_length=MAX_SEQ_LENGTH,

    # Packing is configured here in TRL 1.13.0
    packing=True,
)

# Create the SFT trainer
trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    processing_class=tokenizer,

    args=training_args,
    peft_config=peft_config,
)

print("SFTTrainer initialized.")
print(
    f"Effective batch size: "
    f"{training_args.per_device_train_batch_size * training_args.gradient_accumulation_steps}"
)

# Start training
train_result = trainer.train()

print(f"\nTraining complete!")
print(f"Final loss: {train_result.training_loss:.4f}")

GPU allocated: 2.06 GB
GPU reserved:  2.10 GB


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

SFTTrainer initialized.
Effective batch size: 16


Step,Training Loss
10,2.087164
20,1.840285



Training complete!
Final loss: 1.9443


In [ ]:
import trl
import inspect

print("TRL version:", trl.__version__)
print("SFTConfig signature:")
print(inspect.signature(SFTConfig))

TRL version: 1.13.0
SFTConfig signature:
(output_dir: str | None = None, per_device_train_batch_size: int = 8, num_train_epochs: float = 3.0, max_steps: int = -1, learning_rate: float = 2e-05, lr_scheduler_type: transformers.trainer_utils.SchedulerType | str = 'linear', lr_scheduler_kwargs: dict | str | None = None, warmup_steps: float = 0, optim: transformers.training_args.OptimizerNames | str = 'adamw_torch_fused', optim_args: str | None = None, weight_decay: float = 0.0, adam_beta1: float = 0.9, adam_beta2: float = 0.999, adam_epsilon: float = 1e-08, optim_target_modules: None | str | list[str] = None, gradient_accumulation_steps: int = 1, average_tokens_across_devices: bool = True, max_grad_norm: float = 1.0, label_smoothing_factor: float = 0.0, bf16: bool | None = None, fp16: bool = False, bf16_full_eval: bool = False, fp16_full_eval: bool = False, tf32: bool | None = None, gradient_checkpointing: bool = True, gradient_checkpointing_kwargs: dict[str, typing.Any] | str | None = Non

---
## Part 7 — Save the Adapter

### Saved the LoRA adapter and print its size

1. Saved the trained model to `f"{OUTPUT_DIR}/final-adapter"`.
2. Also save the tokenizer to the same path.
3. Computed and print the adapter size in MB.

In [ ]:
import os

# Path for the final LoRA adapter
adapter_dir = f"{OUTPUT_DIR}/final-adapter"

# Save the trained LoRA adapter
trainer.model.save_pretrained(adapter_dir)

# Save the tokenizer
tokenizer.save_pretrained(adapter_dir)

print(f"LoRA adapter saved to: {adapter_dir}")

# Calculate adapter size
adapter_size_bytes = 0

for root, dirs, files in os.walk(adapter_dir):
    for file in files:
        file_path = os.path.join(root, file)
        adapter_size_bytes += os.path.getsize(file_path)

adapter_size_mb = adapter_size_bytes / (1024 ** 2)

print(f"Adapter size: {adapter_size_mb:.2f} MB")

LoRA adapter saved to: ./sft-lora-qwen/final-adapter
Adapter size: 7.77 MB


---
## Part 8 — Evaluataion

### Task 8: Compared base vs fine-tuned responses

1. Used `trainer.model` (already has LoRA weights) to generate responses for the same `test_prompts`.
2. Printed a side-by-side comparison of base vs fine-tuned responses.

In [ ]:
print("=" * 70)
print("FINE-TUNED MODEL RESPONSES (after SFT)")
print("=" * 70)

for i, prompt in enumerate(test_prompts):
    print(f"\n{'-' * 60}")
    print(f"Prompt {i+1}: {prompt}")
    print(f"{'-' * 60}")

    # Generate response using trained LoRA model
    response = generate_response(
        trainer.model,
        tokenizer,
        prompt,
        max_new_tokens=200
    )

    print(f"Response: {response[:500]}")

    # Compare with base model response
    print(f"\n{'-' * 60}")
    print("Base Model Response:")
    print(f"{'-' * 60}")
    print(base_responses[i][:500])

[transformers] Both `max_new_tokens` (=200) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


FINE-TUNED MODEL RESPONSES (after SFT)

------------------------------------------------------------
Prompt 1: Explain what a neural network is in 2-3 sentences.
------------------------------------------------------------


[transformers] Both `max_new_tokens` (=200) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Response: A neural network is a mathematical model that mimics the functioning of the human brain. It consists of a series of interconnected processing units (neurons) that can receive, process, and transmit information. Neurons can be trained to recognize patterns, solve problems, and make decisions. The model uses layers of neurons, each with a different function, to create an overall functioning neural network.

In simpler terms, a neural network is a mathematical model that mimics the way the human br

------------------------------------------------------------
Base Model Response:
------------------------------------------------------------
A neural network is a type of artificial neural system that simulates the way the human brain processes and makes decisions. It consists of a series of layers of neurons, each connected to the previous layer through synapses. The neurons in the first layer receive input from other neurons, process the data, and output a result or a decision. T

---
## Done

I have successfully fine-tuned TinyLlama with LoRA on the Dolly dataset. Reviewed your base vs fine-tuned outputs and note the differences in instruction-following quality.